# Cognitive LLM — Phase 1 Ablation Sweep

**Purpose:** Run all 10 experiments from `agent.py` on a Colab/Kaggle GPU.

**Experiments (priority order):**
| ID | Config | Hypothesis |
|----|--------|-----------|
| EXP_V00 | No LoRA, no blocks | Vanilla pretrained baseline |
| EXP_000 | LoRA only | Training baseline (3x median) |
| EXP_001 | B6 | HomeostaticNorm alone |
| EXP_002 | B2+B6 | EpisodicMemory (best bet) |
| EXP_003 | B1+B6 | SurpriseGate |
| EXP_004 | B3+B6 | PerLayerCritic (TD) |
| EXP_005 | B1+B2+B6 | Surprise + Memory |
| EXP_006 | B2+B3+B6 | Memory + Critic |
| EXP_007 | B1+B2+B3+B6 | Full Phase 1 stack |
| EXP_008 | B1+B3+B6 | Surprise + Critic |

**Runtime:** ~2 hours on T4, ~1 hour on A100

In [ ]:
# Cell 1: Install dependencies
%pip install -q transformers datasets accelerate bitsandbytes peft trl wandb lm-eval pyyaml matplotlib

In [ ]:
# Cell 2: Clone repo and setup
from pathlib import Path
import os, subprocess, sys

repo_dir = Path('/content/cognitive-llm')

if repo_dir.exists():
    import shutil
    shutil.rmtree(repo_dir)

subprocess.run(
    ['git', 'clone', 'https://github.com/RiyadMehdi7/cognitive-llm.git', str(repo_dir)],
    check=True,
)
os.chdir(repo_dir)

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# Clear cached modules
for name in list(sys.modules):
    if name.startswith('cognitive_llm'):
        del sys.modules[name]

print(f'Working directory: {Path.cwd()}')
print('Ready')

In [ ]:
# Cell 3: Verify GPU and imports
import torch

assert torch.cuda.is_available(), 'GPU runtime required! Go to Runtime > Change runtime type > T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

capability = torch.cuda.get_device_capability(0)
print(f'Compute capability: {capability[0]}.{capability[1]}')
print(f'dtype: {"bf16" if capability[0] >= 8 else "fp16"}')

# Verify train.py imports cleanly
import train
print('train.py OK')

# Verify agent.py
import agent
print(f'agent.py OK — {len(agent.HYPOTHESIS_SPACE)} experiments defined')

## Option A: Run full automated sweep
Runs all 10 experiments via `agent.py`. Takes ~2h on T4, ~1h on A100.
Results go to `results.tsv` and `lab_notebook.md`.

In [ ]:
# Cell 4: Run full experiment sweep via agent.py
# This runs ALL 10 experiments sequentially, each as a subprocess.
# Safe to interrupt — completed experiments are saved to results.tsv
# and will be skipped on re-run.

!python agent.py --run

## Option B: Run a single experiment manually
Use this to test a specific config without running the full sweep.

In [ ]:
# Cell 5: Run a single experiment manually
# Edit train.py CONFIG directly, then run:

# Example: test B2+B6 only
import re

source = Path('train.py').read_text()
overrides = {
    'use_block1': False, 'use_block2': True, 'use_block3': False,
    'use_block4': False, 'use_block5': False, 'use_block6': True,
    'skip_lora': False,
    'max_steps': 200,  # quick test
}
for key, value in overrides.items():
    source = re.sub(
        rf'("{re.escape(key)}"\s*:\s*)([^,}}\n]+)',
        rf'\g<1>{repr(value)}',
        source,
    )
Path('train.py').write_text(source)
print('CONFIG patched. Running...')
!python train.py

## Results Analysis
Run after experiments complete (Option A or B).

In [ ]:
# Cell 6: Load and display results
import csv
from pathlib import Path

results_path = Path('results.tsv')
if not results_path.exists():
    print('No results.tsv found. Run experiments first (Cell 4 or 5).')
else:
    with open(results_path) as f:
        rows = list(csv.DictReader(f, delimiter='\t'))

    print(f"{'ID':<10} {'Hypothesis':<45} {'val_loss':>10} {'gsm8k%':>8} {'delta%':>8} {'time':>6} {'verdict':>8}")
    print('-' * 100)
    for r in rows:
        print(
            f"{r['exp_id']:<10} {r['hypothesis']:<45} "
            f"{r['val_loss']:>10} {r.get('gsm8k_acc','n/a'):>8} "
            f"{r['baseline_delta_pct']:>8} {r['run_minutes']:>6} {r['verdict']:>8}"
        )

    # Find best
    valid = [r for r in rows if r['val_loss'] not in ('crash', '')]
    if valid:
        best = min(valid, key=lambda r: float(r['val_loss']))
        print(f"\nBest: {best['exp_id']} — {best['hypothesis']}")
        print(f"  val_loss: {best['val_loss']}")
        print(f"  gsm8k_acc: {best.get('gsm8k_acc', 'n/a')}%")
        print(f"  improvement: {best['baseline_delta_pct']}% vs LoRA baseline")

In [ ]:
# Cell 7: Plot results
import matplotlib.pyplot as plt
import csv, os
from pathlib import Path

results_path = Path('results.tsv')
assert results_path.exists(), 'Run experiments first'

with open(results_path) as f:
    rows = list(csv.DictReader(f, delimiter='\t'))

# Filter valid results
valid = [r for r in rows if r['val_loss'] not in ('crash', '')]
names = [r['exp_id'] for r in valid]
val_losses = [float(r['val_loss']) for r in valid]
gsm8k_accs = [float(r.get('gsm8k_acc', 0)) if r.get('gsm8k_acc', 'n/a') != 'n/a' else 0 for r in valid]
labels = [r['hypothesis'].split(':')[0].strip() if ':' in r['hypothesis'] else r['hypothesis'][:20] for r in valid]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Val loss comparison
colors = ['#888' if 'Vanilla' in r['hypothesis'] or 'Baseline' in r['hypothesis']
          else '#2563a0' for r in valid]
bars1 = ax1.bar(range(len(names)), val_losses, color=colors, alpha=0.85)
ax1.set_xticks(range(len(names)))
ax1.set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
ax1.set_ylabel('Val Loss')
ax1.set_title('Validation Loss by Configuration')
ax1.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, val in zip(bars1, val_losses):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', fontsize=8)

# Baseline line
baseline = next((float(r['val_loss']) for r in valid if r['exp_id'] == 'EXP_000'), None)
if baseline:
    ax1.axhline(y=baseline, color='red', linestyle='--', alpha=0.5, label=f'LoRA baseline ({baseline:.3f})')
    ax1.legend(fontsize=8)

# GSM8K accuracy
bars2 = ax2.bar(range(len(names)), gsm8k_accs, color=colors, alpha=0.85)
ax2.set_xticks(range(len(names)))
ax2.set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
ax2.set_ylabel('GSM8K Accuracy (%)')
ax2.set_title('GSM8K Accuracy (50 test samples)')
ax2.grid(True, alpha=0.3, axis='y')

for bar, acc in zip(bars2, gsm8k_accs):
    if acc > 0:
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{acc:.1f}%', ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('ablation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ablation_results.png')

In [ ]:
# Cell 8: Improvement analysis
import csv
from pathlib import Path

with open('results.tsv') as f:
    rows = list(csv.DictReader(f, delimiter='\t'))

valid = [r for r in rows if r['val_loss'] not in ('crash', '')]
baseline_row = next((r for r in valid if r['exp_id'] == 'EXP_000'), None)
vanilla_row = next((r for r in valid if r['exp_id'] == 'EXP_V00'), None)

if baseline_row:
    bl = float(baseline_row['val_loss'])
    print(f"LoRA Baseline: val_loss={bl:.6f}")
    if vanilla_row:
        vl = float(vanilla_row['val_loss'])
        print(f"Vanilla (no LoRA): val_loss={vl:.6f}")
        print(f"LoRA improvement over vanilla: {(vl - bl) / vl * 100:.1f}%")

    print(f"\n{'Experiment':<45} {'val_loss':>10} {'delta%':>8} {'gsm8k':>8} {'verdict':>8}")
    print('-' * 85)
    for r in valid:
        vl = float(r['val_loss'])
        delta = (bl - vl) / bl * 100 if r['exp_id'] != 'EXP_V00' else 0
        marker = ' <-- BEST' if vl == min(float(x['val_loss']) for x in valid if x['exp_id'] != 'EXP_V00') and r['exp_id'] != 'EXP_V00' else ''
        print(
            f"{r['hypothesis']:<45} {r['val_loss']:>10} {delta:>+7.2f}% "
            f"{r.get('gsm8k_acc','n/a'):>8} {r['verdict']:>8}{marker}"
        )

    # Key findings
    non_baseline = [r for r in valid if r['exp_id'] not in ('EXP_000', 'EXP_V00')]
    if non_baseline:
        best = min(non_baseline, key=lambda r: float(r['val_loss']))
        best_delta = (bl - float(best['val_loss'])) / bl * 100
        print(f"\nBest config: {best['hypothesis']}")
        print(f"  val_loss improvement: {best_delta:+.2f}% vs LoRA baseline")
        if best_delta > 5:
            print("  STATUS: TARGET MET (>5% improvement)")
        elif best_delta > 0:
            print(f"  STATUS: Positive signal, need more training steps for Phase 2")
        else:
            print("  STATUS: No improvement — consider negative ablation result")

In [ ]:
# Cell 9: Download results to local machine
# In Colab: this triggers a browser download
# In Kaggle: files appear in the output tab

import shutil
from pathlib import Path

# Bundle results
output_dir = Path('phase1_results')
output_dir.mkdir(exist_ok=True)

for f in ['results.tsv', 'lab_notebook.md', 'ablation_results.png']:
    if Path(f).exists():
        shutil.copy(f, output_dir / f)

if Path('experiments').exists():
    shutil.copytree('experiments', output_dir / 'experiments', dirs_exist_ok=True)

# Create zip
shutil.make_archive('phase1_results', 'zip', '.', 'phase1_results')
print('Created: phase1_results.zip')

# Colab download
try:
    from google.colab import files
    files.download('phase1_results.zip')
    print('Download started in browser')
except ImportError:
    print('Not in Colab — find phase1_results.zip in the file browser')

## Experiment Logs
View individual experiment logs if something crashed.

In [ ]:
# Cell 10: View a specific experiment log
# Change the exp_id to view different logs

exp_id = 'exp_002'  # change this: exp_v00, exp_000, exp_001, ..., exp_008
log_path = Path(f'experiments/{exp_id}.log')

if log_path.exists():
    text = log_path.read_text()
    # Show last 50 lines (most relevant)
    lines = text.strip().split('\n')
    if len(lines) > 50:
        print(f'... ({len(lines) - 50} lines omitted) ...\n')
    print('\n'.join(lines[-50:]))
else:
    print(f'No log found at {log_path}')
    print('Available logs:')
    for p in sorted(Path('experiments').glob('*.log')):
        print(f'  {p.name}')